# 03 — Siam-GCAN (Siamese Graph Convolutional Attention Network)

**Family:** Graph CNN + Siamese &nbsp;•&nbsp; **Input:** DE feature pairs `[14 x 5]`

A twin-branch, weight-shared GCN encoder with multi-head self-attention over the
electrode graph. Training uses **trial pairs**: the contrastive term pulls
same-emotion windows together and pushes different-emotion windows apart in the
embedding space, on top of a per-branch cross-entropy term. Inference uses a
single branch, so the shared evaluation harness works unchanged.

## 1. Setup

In [ ]:
import os, sys

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src import benchmark_si as B
from src.evaluation import set_seed, get_device, count_parameters
from src.models import siam_gcan

set_seed(42)
DEVICE = get_device()

# ----- experiment configuration -----
MODEL_NAME = "SiamGCAN"
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> 3-fold / few-epoch smoke test
                      # False -> the full 23-fold benchmark used for the thesis table
print("device:", DEVICE, "| model:", MODEL_NAME, "| label:", LABEL, "| QUICK:", QUICK)
print("recipe:", B.REGISTRY[MODEL_NAME])


## 2. Architecture

`src/models/siam_gcan.py`: `GCANEncoder` = 2 × (`GraphAttnLayer` = learnable-adjacency
GCN propagation + `nn.MultiheadAttention` over electrodes + residual LayerNorm),
mean-pooled over nodes and projected to a 64-D embedding. A small MLP head maps the
embedding to class logits.

**Pair sampling** (`SiamesePairDataset`) draws, for each window, a partner that is
same-class ~50% of the time; `pair_label = 1` iff the two share an emotion class.
Pairs are resampled every epoch.

In [ ]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (14, 5)}
_m = siam_gcan.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

## 3. Run the subject-independent (LOSO) benchmark

The benchmark uses a single protocol: **23-fold leave-one-subject-out**
(`data/splits/loso_splits.json`). Fold *k* trains a fresh model on 22 subjects and
tests on the held-out 23rd; normalisation is fit on the training subjects only. We
report the **mean ± std across the 23 folds**.

Siam-GCAN's paired objective (`CE(a) + CE(b) + λ·contrastive(emb_a, emb_b)` with
`SiamesePairDataset`) cannot be expressed by the default training loop, so its
custom `fit_fn` lives in `src/benchmark_si.py::make_siam_fit`. `run_one` picks it up
automatically from this model's `REGISTRY` entry — the notebook just calls `run_one`.


In [ ]:
# One shared call: 23-fold LOSO (Siamese-pair training), then writes results/SiamGCAN_results.json
res = B.run_one(MODEL_NAME, label=LABEL, quick=QUICK, device=DEVICE)
res

## Notes for the methodology chapter

* Every metric comes from `src/evaluation.compute_metrics` (accuracy, macro
  precision / recall / F1, confusion matrix) — no per-notebook metric code.
* The Siamese training loop and this model's LOSO recipe (feature view, batch
  size, epochs, lr, `pairs_per_sample`, `contrastive_lambda`) live in
  `src/benchmark_si.py` — a single source of truth shared with the CLI.
* `params_count` and `train_time_sec` are logged to
  `results/{MODEL_NAME}_results.json` for the accuracy-vs-efficiency analysis.
* Set `QUICK = True` to smoke-test (3 folds / few epochs); `LABEL` switches the
  target between valence / arousal / dominance with no other change.
